# SQL/PGQ Graph Retrieval with OraclePageIndex

<p align="center"><b>Structured Retrieval &middot; Entity Search &middot; Relationship Discovery &middot; Multi-Hop Traversal</b></p>

---

## Introduction

Traditional RAG returns chunks ranked by semantic similarity. OraclePageIndex returns
**structured results** from a knowledge graph using SQL/PGQ — Oracle's implementation
of the SQL:2023 Property Graph standard.

This notebook shows how to use the graph for **agentic retrieval**: precise, structured
queries that return entities, relationships, sections, and their connections.

### What you'll learn:
- [x] Entity-based section retrieval
- [x] Cross-entity relationship discovery
- [x] Multi-hop graph traversal with SQL/PGQ
- [x] Raw GRAPH_TABLE queries for custom retrieval

> **Prerequisite:** Run the [Oracle Graph Quickstart](oracle_graph_quickstart.ipynb) notebook first,
> or index a document with `python run.py index demo/apple-10k-2024.pdf`

---

## Setup

In [1]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))

from oracle_pageindex.db import OracleDB
from oracle_pageindex.graph import GraphStore
from oracle_pageindex.utils import ConfigLoader

cfg = ConfigLoader().load()
db = OracleDB(user=cfg.oracle_user, password=cfg.oracle_password, dsn=cfg.oracle_dsn)
db.connect()
graph = GraphStore(db)

# Verify data exists
docs = graph.get_all_documents()
print(f"Connected. Found {len(docs)} indexed document(s):")
for d in docs:
    print(f"  - {d['doc_name']} (doc_id={d['doc_id']})")

Connected. Found 0 indexed document(s):


## 1. Entity-Based Section Retrieval

Find all sections that mention a given entity. This is the graph equivalent of
"give me all context related to X" — but instead of cosine similarity, it follows
actual `mentions` edges in the graph.

In [2]:
# Find sections mentioning a concept
entity_name = "supply chain"

sections = graph.get_entity_sections(entity_name)
print(f'Sections mentioning "{entity_name}":\n')
for s in sections:
    print(f"  [{s['section_id']}] {s['title']}")
    print(f"       Doc: {s['doc_name']} | Relevance: {s['relevance']}")
    if s.get('summary'):
        summary = str(s['summary'])[:100] + '...'
        print(f"       Summary: {summary}")
    print()

Sections mentioning "supply chain":



In [3]:
# Try different entities
for name in ["Tim Cook", "iPhone", "Cybersecurity", "China"]:
    results = graph.get_entity_sections(name)
    print(f'"{name}": {len(results)} section(s)')

"Tim Cook": 0 section(s)
"iPhone": 0 section(s)
"Cybersecurity": 0 section(s)
"China": 0 section(s)


## 2. Entity Relationship Discovery

Discover how entities connect to each other through `related_to` edges.
These relationships were extracted by the LLM during indexing.

In [4]:
# Find entities related to Apple
related = graph.get_related_entities("Apple")

print("Entities related to Apple:\n")
for r in related:
    name = r.get('name') or r.get('related_name', '?')
    etype = r.get('entity_type') or r.get('related_type', '?')
    rel = r.get('relationship', 'RELATED_TO')
    print(f"  Apple --[{rel}]--> {name} ({etype})")

Entities related to Apple:



## 3. SQL/PGQ: GRAPH_TABLE Queries

The `doc_knowledge_graph` Property Graph is queryable with standard SQL using
`GRAPH_TABLE(...MATCH...)`. Below are examples of increasing complexity.

### 3.1 Entity → Section traversal

In [5]:
# SQL/PGQ: Find sections mentioning a specific entity
results = graph.graph_query_entity_sections("Risk Factors")

print('SQL/PGQ: Sections mentioning "Risk Factors":\n')
for r in results:
    print(f"  Section: {r['section_title']} (depth={r['depth_level']})")
    print(f"  Entity:  {r['entity_name']} ({r['entity_type']})")
    print(f"  Relevance: {r['relevance']}")
    print()

SQL/PGQ: Sections mentioning "Risk Factors":



### 3.2 Entity → Entity traversal

In [6]:
# SQL/PGQ: Find all entity-to-entity relationships
results = graph.graph_query_related_entities("Apple Inc.")

print('SQL/PGQ: Entities connected to "Apple Inc.":\n')
for r in results:
    print(f"  {r['source_name']} --[{r['relationship']}]--> {r['related_name']} ({r['related_type']})")

SQL/PGQ: Entities connected to "Apple Inc.":



### 3.3 Multi-hop: Recursive section hierarchy

In [7]:
# Find ALL descendants of a section (multi-hop traversal)
# Uses Oracle CONNECT BY PRIOR for recursive hierarchy traversal

# First, find a top-level section title to query
top_sections = db.fetchall("""
    SELECT title FROM sections WHERE depth_level = 0 AND title IS NOT NULL
    FETCH FIRST 5 ROWS ONLY
""")
print("Top-level sections:")
for s in top_sections:
    print(f"  - {s['title']}")

# Recursive traversal
if top_sections:
    section_title = top_sections[0]['title']
    children = graph.graph_query_section_children(section_title)
    print(f'\nDescendants of "{section_title}":\n')
    for c in children:
        indent = "  " * c.get('child_depth', 0)
        print(f"  {indent}{c['child_title']} (depth={c['child_depth']})")

Top-level sections:


### 3.4 Custom raw SQL/PGQ queries

In [8]:
# Count entities by type
entity_stats = db.fetchall("""
    SELECT entity_type, COUNT(*) as cnt
    FROM entities
    GROUP BY entity_type
    ORDER BY cnt DESC
""")

print("Entity type distribution:\n")
for r in entity_stats:
    print(f"  {r['entity_type']:20s} {r['cnt']:4d}")

Entity type distribution:



In [9]:
# Most-mentioned entities (SQL/PGQ GRAPH_TABLE + aggregation)
popular = db.fetchall("""
    SELECT entity_name, entity_type, COUNT(*) AS mention_count
    FROM (
        SELECT *
        FROM GRAPH_TABLE (doc_knowledge_graph
            MATCH (s IS section) -[m IS mentions]-> (e IS entity)
            COLUMNS (
                e.name AS entity_name,
                e.entity_type AS entity_type
            )
        )
    )
    GROUP BY entity_name, entity_type
    ORDER BY mention_count DESC
    FETCH FIRST 15 ROWS ONLY
""")

print("Most-mentioned entities (SQL/PGQ):\n")
for r in popular:
    print(f"  {r['entity_name']:30s} ({r['entity_type']:12s}) - {r['mention_count']} mentions")

Most-mentioned entities (SQL/PGQ):



## 4. Structured Retrieval for Agents

Unlike vector RAG that returns opaque chunks, graph retrieval returns structured data
that agents can reason over programmatically.

In [10]:
def structured_retrieve(entity_name):
    """Retrieve structured context for an entity — suitable for agent consumption."""
    sections = graph.get_entity_sections(entity_name)
    related = graph.get_related_entities(entity_name)

    return {
        "entity": entity_name,
        "sections": [
            {
                "title": s.get("title", ""),
                "doc_name": s.get("doc_name", ""),
                "relevance": s.get("relevance", ""),
                "summary": str(s.get("summary", ""))[:200],
            }
            for s in sections
        ],
        "related_entities": [
            {
                "name": r.get("name") or r.get("related_name", ""),
                "type": r.get("entity_type") or r.get("related_type", ""),
                "relationship": r.get("relationship", "RELATED_TO"),
            }
            for r in related
        ],
    }


result = structured_retrieve("Cybersecurity")
print(json.dumps(result, indent=2, default=str))

{
  "entity": "Cybersecurity",
  "sections": [],
  "related_entities": []
}


## Cleanup

In [11]:
db.close()
print("Oracle connection closed")

Oracle connection closed


---

## Summary

This notebook demonstrated **structured graph retrieval** using Oracle SQL/PGQ:

| Method | What it does |
|--------|-------------|
| `get_entity_sections()` | Follow `mentions` edges to find sections |
| `get_related_entities()` | Follow `related_to` edges between entities |
| `graph_query_entity_sections()` | Same via SQL/PGQ `GRAPH_TABLE` |
| `graph_query_related_entities()` | Same via SQL/PGQ `GRAPH_TABLE` |
| `graph_query_section_children()` | Recursive CONNECT BY hierarchy traversal |
| Raw `db.fetchall()` | Any custom SQL/PGQ query |

Every query returns **structured, traceable results** — not opaque similarity scores.

---

*Built with [OraclePageIndex](https://github.com/jasperan/OraclePageIndex) — Oracle AI Database powered document intelligence with Property Graphs.*